In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression

In [4]:
df = pd.read_csv("autonomous_driving_expanded_dataset.csv")

In [5]:
df.head()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,ego_acceleration_mps2,steering_angle_deg,yaw_rate_rads,throttle_position,brake_pressure,weather_condition,visibility_range_m,road_surface_condition,behavior_label
0,38.079472,-3.190934,6,0.675600,48.665354,0.741882,-0.012507,3.073696,80,26.231337,-1.898323,-6.714563,0.399968,0.051580,0.325341,rain,213.167374,wet,follow
1,95.120716,-0.796930,3,0.620099,49.519209,0.682938,-0.036788,3.449886,50,15.299050,0.930789,-24.890723,-0.150016,0.627191,0.361242,fog,227.492200,icy,follow
2,73.467400,10.636422,1,-0.216601,32.508906,0.849911,-0.026277,3.982308,30,7.015695,2.439927,14.601049,-0.436415,0.825701,0.161016,night,291.796782,icy,follow
3,60.267190,-4.799868,4,-0.613688,6.355965,0.215723,0.014275,4.179263,40,27.361192,-0.386392,23.101919,0.333522,0.556257,0.083264,rain,70.354785,icy,follow
4,16.445845,11.089491,13,1.343696,25.308504,0.753826,-0.044812,4.112925,40,23.701309,2.548616,-20.299001,0.414422,0.320478,0.578371,night,279.849029,icy,yield


In [6]:
df.shape

(5000, 19)

In [7]:
df.isnull().sum()

obstacle_distance_m           0
relative_speed_mps            0
num_obstacles                 0
lane_offset_m                 0
traffic_density_veh_per_km    0
risk_probability              0
road_curvature_1pm            0
road_width_m                  0
speed_limit_kmh               0
ego_speed_mps                 0
ego_acceleration_mps2         0
steering_angle_deg            0
yaw_rate_rads                 0
throttle_position             0
brake_pressure                0
weather_condition             0
visibility_range_m            0
road_surface_condition        0
behavior_label                0
dtype: int64

In [8]:
df.behavior_label.value_counts()

behavior_label
follow         2812
stop            893
lane_change     739
yield           323
overtake        233
Name: count, dtype: int64

In [9]:
#OneHotEncodingForWeatherCondition

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

weather_ohe = ohe.fit_transform(df[['weather_condition']])

weather_df = pd.DataFrame(weather_ohe,columns=ohe.get_feature_names_out(['weather_condition']))

df = pd.concat([df.reset_index(drop=True),weather_df],axis=1)
df = df.drop('weather_condition',axis=1)

df.head()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,...,yaw_rate_rads,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain
0,38.079472,-3.190934,6,0.675600,48.665354,0.741882,-0.012507,3.073696,80,26.231337,...,0.399968,0.051580,0.325341,213.167374,wet,follow,0.0,0.0,0.0,1.0
1,95.120716,-0.796930,3,0.620099,49.519209,0.682938,-0.036788,3.449886,50,15.299050,...,-0.150016,0.627191,0.361242,227.492200,icy,follow,0.0,1.0,0.0,0.0
2,73.467400,10.636422,1,-0.216601,32.508906,0.849911,-0.026277,3.982308,30,7.015695,...,-0.436415,0.825701,0.161016,291.796782,icy,follow,0.0,0.0,1.0,0.0
3,60.267190,-4.799868,4,-0.613688,6.355965,0.215723,0.014275,4.179263,40,27.361192,...,0.333522,0.556257,0.083264,70.354785,icy,follow,0.0,0.0,0.0,1.0
4,16.445845,11.089491,13,1.343696,25.308504,0.753826,-0.044812,4.112925,40,23.701309,...,0.414422,0.320478,0.578371,279.849029,icy,yield,0.0,0.0,1.0,0.0


In [10]:
road_condition_mapping = {'dry':3,'wet':2,'icy':1}
df['road_surface_condition'] = df['road_surface_condition'].map(road_condition_mapping)

df.head()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,...,yaw_rate_rads,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain
0,38.079472,-3.190934,6,0.675600,48.665354,0.741882,-0.012507,3.073696,80,26.231337,...,0.399968,0.051580,0.325341,213.167374,2,follow,0.0,0.0,0.0,1.0
1,95.120716,-0.796930,3,0.620099,49.519209,0.682938,-0.036788,3.449886,50,15.299050,...,-0.150016,0.627191,0.361242,227.492200,1,follow,0.0,1.0,0.0,0.0
2,73.467400,10.636422,1,-0.216601,32.508906,0.849911,-0.026277,3.982308,30,7.015695,...,-0.436415,0.825701,0.161016,291.796782,1,follow,0.0,0.0,1.0,0.0
3,60.267190,-4.799868,4,-0.613688,6.355965,0.215723,0.014275,4.179263,40,27.361192,...,0.333522,0.556257,0.083264,70.354785,1,follow,0.0,0.0,0.0,1.0
4,16.445845,11.089491,13,1.343696,25.308504,0.753826,-0.044812,4.112925,40,23.701309,...,0.414422,0.320478,0.578371,279.849029,1,yield,0.0,0.0,1.0,0.0


In [11]:
le = LabelEncoder()

df['behavior_label'] = le.fit_transform(df[['behavior_label']])



/opt/anaconda3/envs/streamlit_env/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [12]:
df.head()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,...,yaw_rate_rads,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain
0,38.079472,-3.190934,6,0.675600,48.665354,0.741882,-0.012507,3.073696,80,26.231337,...,0.399968,0.051580,0.325341,213.167374,2,0,0.0,0.0,0.0,1.0
1,95.120716,-0.796930,3,0.620099,49.519209,0.682938,-0.036788,3.449886,50,15.299050,...,-0.150016,0.627191,0.361242,227.492200,1,0,0.0,1.0,0.0,0.0
2,73.467400,10.636422,1,-0.216601,32.508906,0.849911,-0.026277,3.982308,30,7.015695,...,-0.436415,0.825701,0.161016,291.796782,1,0,0.0,0.0,1.0,0.0
3,60.267190,-4.799868,4,-0.613688,6.355965,0.215723,0.014275,4.179263,40,27.361192,...,0.333522,0.556257,0.083264,70.354785,1,0,0.0,0.0,0.0,1.0
4,16.445845,11.089491,13,1.343696,25.308504,0.753826,-0.044812,4.112925,40,23.701309,...,0.414422,0.320478,0.578371,279.849029,1,4,0.0,0.0,1.0,0.0


In [13]:
df.describe()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,...,yaw_rate_rads,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,...,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,50.186367,-0.255386,6.955400,0.010073,25.583395,0.499549,-0.000604,3.746310,51.858000,15.062469,...,0.004099,0.507082,0.495452,274.716370,1.999600,1.035200,0.260200,0.249000,0.238400,0.252400
std,28.673731,8.568495,4.360323,1.156500,14.487988,0.286204,0.028783,0.716901,17.383563,8.675463,...,0.289647,0.291403,0.290383,130.923487,0.811253,1.379834,0.438787,0.432477,0.426148,0.434433
min,1.001152,-14.998415,0.000000,-1.998990,0.002406,0.000110,-0.049993,2.500014,30.000000,0.002798,...,-0.499952,0.000209,0.000115,50.078817,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,25.142418,-7.585623,3.000000,-0.995584,13.306720,0.253389,-0.025865,3.120841,40.000000,7.530667,...,-0.247637,0.258932,0.244703,161.771167,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,50.500854,-0.420853,7.000000,0.023356,25.773559,0.500906,-0.000717,3.755055,50.000000,15.165525,...,0.004641,0.502534,0.491657,271.512040,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,75.061985,7.001148,11.000000,1.018979,38.023769,0.740077,0.024353,4.359845,60.000000,22.628872,...,0.258458,0.766136,0.750211,388.628445,3.000000,2.000000,1.000000,0.000000,0.000000,1.000000
max,99.972050,14.985156,14.000000,1.999699,49.995049,0.999598,0.049973,4.999473,80.000000,29.993913,...,0.499895,0.999624,0.999677,499.949029,3.000000,4.000000,1.000000,1.000000,1.000000,1.000000


In [14]:
df['speed_limit_mps'] = df['speed_limit_kmh']/3.6
df = df.drop('speed_limit_kmh',axis=1)
df.head()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,ego_speed_mps,ego_acceleration_mps2,...,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain,speed_limit_mps
0,38.079472,-3.190934,6,0.675600,48.665354,0.741882,-0.012507,3.073696,26.231337,-1.898323,...,0.051580,0.325341,213.167374,2,0,0.0,0.0,0.0,1.0,22.222222
1,95.120716,-0.796930,3,0.620099,49.519209,0.682938,-0.036788,3.449886,15.299050,0.930789,...,0.627191,0.361242,227.492200,1,0,0.0,1.0,0.0,0.0,13.888889
2,73.467400,10.636422,1,-0.216601,32.508906,0.849911,-0.026277,3.982308,7.015695,2.439927,...,0.825701,0.161016,291.796782,1,0,0.0,0.0,1.0,0.0,8.333333
3,60.267190,-4.799868,4,-0.613688,6.355965,0.215723,0.014275,4.179263,27.361192,-0.386392,...,0.556257,0.083264,70.354785,1,0,0.0,0.0,0.0,1.0,11.111111
4,16.445845,11.089491,13,1.343696,25.308504,0.753826,-0.044812,4.112925,23.701309,2.548616,...,0.320478,0.578371,279.849029,1,4,0.0,0.0,1.0,0.0,11.111111


In [15]:
df.describe()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,ego_speed_mps,ego_acceleration_mps2,...,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain,speed_limit_mps
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,...,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,50.186367,-0.255386,6.955400,0.010073,25.583395,0.499549,-0.000604,3.746310,15.062469,0.021161,...,0.507082,0.495452,274.716370,1.999600,1.035200,0.260200,0.249000,0.238400,0.252400,14.405000
std,28.673731,8.568495,4.360323,1.156500,14.487988,0.286204,0.028783,0.716901,8.675463,1.723090,...,0.291403,0.290383,130.923487,0.811253,1.379834,0.438787,0.432477,0.426148,0.434433,4.828768
min,1.001152,-14.998415,0.000000,-1.998990,0.002406,0.000110,-0.049993,2.500014,0.002798,-2.998075,...,0.000209,0.000115,50.078817,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.333333
25%,25.142418,-7.585623,3.000000,-0.995584,13.306720,0.253389,-0.025865,3.120841,7.530667,-1.460144,...,0.258932,0.244703,161.771167,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.111111
50%,50.500854,-0.420853,7.000000,0.023356,25.773559,0.500906,-0.000717,3.755055,15.165525,0.014376,...,0.502534,0.491657,271.512040,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,13.888889
75%,75.061985,7.001148,11.000000,1.018979,38.023769,0.740077,0.024353,4.359845,22.628872,1.509086,...,0.766136,0.750211,388.628445,3.000000,2.000000,1.000000,0.000000,0.000000,1.000000,16.666667
max,99.972050,14.985156,14.000000,1.999699,49.995049,0.999598,0.049973,4.999473,29.993913,2.998854,...,0.999624,0.999677,499.949029,3.000000,4.000000,1.000000,1.000000,1.000000,1.000000,22.222222


In [16]:
num_cols = [
    'obstacle_distance_m', 'relative_speed_mps', 'num_obstacles', 
    'lane_offset_m', 'traffic_density_veh_per_km', 'risk_probability', 
    'road_curvature_1pm', 'road_width_m', 'speed_limit_mps', 
    'ego_speed_mps', 'ego_acceleration_mps2', 'steering_angle_deg', 
    'yaw_rate_rads', 'throttle_position', 'brake_pressure', 
    'visibility_range_m'
]

scaler = StandardScaler()

df[num_cols]=scaler.fit_transform(df[num_cols])

df.head()


,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,ego_speed_mps,ego_acceleration_mps2,...,throttle_position,brake_pressure,visibility_range_m,road_surface_condition,behavior_label,weather_condition_clear,weather_condition_fog,weather_condition_night,weather_condition_rain,speed_limit_mps
0,-0.422272,-0.342632,-0.219134,0.575524,1.593338,0.846798,-0.413598,-0.938317,1.287538,-1.114089,...,-1.563292,-0.585871,-0.470161,2,0,0.0,0.0,0.0,1.0,1.619047
1,1.567248,-0.063208,-0.907225,0.527528,1.652280,0.640825,-1.257276,-0.413520,0.027273,0.527958,...,0.412216,-0.462228,-0.360737,1,0,0.0,1.0,0.0,0.0,-0.106893
2,0.812010,1.271273,-1.365953,-0.196020,0.478065,1.224291,-0.892039,0.329225,-0.927625,1.403878,...,1.093504,-1.151818,0.130474,1,0,0.0,0.0,1.0,0.0,-1.257520
3,0.351605,-0.530424,-0.677862,-0.539406,-1.327262,-0.991790,0.516961,0.603984,1.417786,-0.236548,...,0.168770,-1.419603,-1.561080,1,0,0.0,0.0,0.0,1.0,-0.682207
4,-1.176823,1.324154,1.386412,1.153269,-0.018976,0.888535,-1.536076,0.511440,0.995878,1.466962,...,-0.640428,0.285579,0.039207,1,4,0.0,0.0,1.0,0.0,-0.682207


In [17]:
X = df.drop('behavior_label',axis=1)
y = df['behavior_label']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
sm = SMOTE(random_state=12)

X_res, y_res = sm.fit_resample(X_train,y_train)

X_res.shape, y_res.shape

((11170, 21), (11170,))

In [19]:
y_res.value_counts()

behavior_label
0    2234
2    2234
1    2234
3    2234
4    2234
Name: count, dtype: int64

In [22]:
#Training_DecisionTree_With_Original_Dataset

dt_model = DecisionTreeClassifier( class_weight='balanced', random_state=42)

dt_model.fit(X_train,y_train)

y_pred = dt_model.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       578
           1       1.00      0.99      1.00       141
           2       1.00      0.98      0.99        43
           3       0.99      1.00      0.99       166
           4       1.00      1.00      1.00        72

    accuracy                           1.00      1000
   macro avg       1.00      0.99      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [ ]:
#Training_DecisionTree_With_Resampled_Dataset

dt_model1 = DecisionTreeClassifier( random_state=42)

dt_model1.fit(X_res,y_res)

y_pred1 = dt_model1.predict(X_test)

print(classification_report(y_test, y_pred1))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       578
           1       0.99      0.98      0.98       141
           2       0.97      0.77      0.86        43
           3       0.89      1.00      0.94       166
           4       0.97      0.92      0.94        72

    accuracy                           0.97      1000
   macro avg       0.96      0.93      0.94      1000
weighted avg       0.97      0.97      0.97      1000



In [ ]:
mapping = dict(zip(range(len(le.classes_)), le.classes_))
print("Class Mapping:", mapping)

Class Mapping: {0: 'follow', 1: 'lane_change', 2: 'overtake', 3: 'stop', 4: 'yield'}


In [ ]:
train_score = dt_model.score(X_train, y_train)
test_score = dt_model.score(X_test, y_test)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Testing Accuracy: {test_score:.4f}")
print(f"Gap: {train_score - test_score:.4f}")

train_score = dt_model1.score(X_train, y_train)
test_score = dt_model1.score(X_test, y_test)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Testing Accuracy: {test_score:.4f}")
print(f"Gap: {train_score - test_score:.4f}")

Training Accuracy: 1.0000
Testing Accuracy: 0.9970
Gap: 0.0030
Training Accuracy: 1.0000
Testing Accuracy: 0.9730
Gap: 0.0270


In [23]:
#Training_RandomForest_With_Org_Dataset

rf_model = RandomForestClassifier(n_estimators=100, max_depth=None, class_weight='balanced',random_state=42)
rf_model.fit(X_train,y_train)

y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99       578
           1       1.00      0.99      0.99       141
           2       1.00      0.91      0.95        43
           3       0.98      1.00      0.99       166
           4       1.00      0.90      0.95        72

    accuracy                           0.99      1000
   macro avg       0.99      0.96      0.97      1000
weighted avg       0.99      0.99      0.99      1000



In [ ]:
#Training_RandomForest_With_Resampled_Dataset

rf_model1 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model1.fit(X_res,y_res)

y_pred_rf1 = rf_model1.predict(X_test)

print(classification_report(y_test, y_pred_rf1))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       578
           1       1.00      0.99      0.99       141
           2       0.76      0.91      0.83        43
           3       0.97      0.92      0.94       166
           4       0.97      0.93      0.95        72

    accuracy                           0.97      1000
   macro avg       0.94      0.95      0.94      1000
weighted avg       0.98      0.97      0.98      1000



In [25]:
#Training_AdaBoost_With_Original_Dataset

from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train, y_train)

y_pred_ada = ada.predict(X_test)

print(classification_report(y_test, y_pred_ada))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97       578
           1       1.00      0.99      1.00       141
           2       1.00      0.98      0.99        43
           3       0.99      1.00      0.99       166
           4       1.00      0.61      0.76        72

    accuracy                           0.97      1000
   macro avg       0.99      0.92      0.94      1000
weighted avg       0.97      0.97      0.97      1000



In [ ]:
#Training_AdaBoost_With_Resampled_Dataset

ada1 = AdaBoostClassifier(n_estimators=100, random_state=42)
ada1.fit(X_res, y_res)

y_pred_ada1 = ada.predict(X_test)

print(classification_report(y_test, y_pred_ada1))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97       578
           1       1.00      0.99      1.00       141
           2       1.00      0.98      0.99        43
           3       0.99      1.00      0.99       166
           4       1.00      0.61      0.76        72

    accuracy                           0.97      1000
   macro avg       0.99      0.92      0.94      1000
weighted avg       0.97      0.97      0.97      1000



In [ ]:
print(f"RF Train Accuracy: {rf_model1.score(X_train, y_train):.4f}")
print(f"RF Test Accuracy: {rf_model1.score(X_test, y_test):.4f}")

RF Train Accuracy: 0.9995
RF Test Accuracy: 0.9750


In [30]:
log_model = LogisticRegression()

log_model.fit(X_train,y_train)

y_pred = log_model.predict(X_test)

print(accuracy_score(y_test,y_pred))
print(classification_report(y_test, y_pred))

0.78
              precision    recall  f1-score   support

           0       0.87      0.90      0.89       578
           1       0.83      0.77      0.80       141
           2       0.57      0.65      0.61        43
           3       0.55      0.64      0.59       166
           4       0.56      0.21      0.30        72

    accuracy                           0.78      1000
   macro avg       0.68      0.63      0.64      1000
weighted avg       0.78      0.78      0.77      1000



In [ ]:
st_kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state =42)

cv_scores = cross_val_score(dt_model, X, y, cv=st_kfolds, scoring='accuracy')

print(f"All Fold Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

All Fold Scores: [0.998 0.996 0.994 0.995 0.999]
Mean CV Accuracy: 0.9964
Standard Deviation: 0.0019


In [26]:
st_kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state =42)

cv_scores = cross_val_score(rf_model, X, y, cv=st_kfolds, scoring='accuracy')

print(f"All Fold Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

All Fold Scores: [0.991 0.992 0.989 0.988 0.996]
Mean CV Accuracy: 0.9912
Standard Deviation: 0.0028


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'bootstrap': [True, False]
}

rf_model2 = RandomForestClassifier(class_weight='balanced', random_state=42)

random_search = RandomizedSearchCV(
    rf_model2, 
    param_distributions=param_dist, 
    n_iter=10, 
    cv=5, 
    random_state=42,
)

random_search.fit(X_train, y_train)

print(f"Best Parameters: {random_search.best_params_}")
best_rf = random_search.best_estimator_

Best Parameters: {'n_estimators': 200, 'min_samples_split': 10, 'max_depth': 30, 'bootstrap': False}


In [28]:
rf_model3 = RandomForestClassifier(n_estimators= 200, min_samples_split= 10, max_depth= 30, bootstrap= False, class_weight='balanced', random_state=42)

rf_model3.fit(X_train,y_train)

y_pred_rf3 = rf_model3.predict(X_test)

print(classification_report(y_test,y_pred_rf3))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       578
           1       1.00      0.99      1.00       141
           2       1.00      0.98      0.99        43
           3       0.99      1.00      0.99       166
           4       1.00      0.99      0.99        72

    accuracy                           1.00      1000
   macro avg       1.00      0.99      0.99      1000
weighted avg       1.00      1.00      1.00      1000



In [29]:
st_kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state =42)

cv_scores = cross_val_score(rf_model3, X, y, cv=st_kfolds, scoring='accuracy')

print(f"All Fold Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

All Fold Scores: [0.995 0.998 0.998 0.993 0.999]
Mean CV Accuracy: 0.9966
Standard Deviation: 0.0022
